# Code for Training Hate Speech Detection models on ULI dataset (Hindi)
**Sentences in Hindi script** | COL8381 Project

Dataset used:
- **Training**: `train_hi_l1.csv`, `train_hi_l2.csv`, `train_hi_l3.csv`
- **Testing** : `test_hi_l1.csv`,  `test_hi_l2.csv`,  `test_hi_l3.csv`

In [1]:
# ── 1. INSTALL & IMPORTS ───────────────────────────────────────────────────
!pip install -q transformers torch datasets lime shap kaggle

import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from collections import defaultdict
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_score, recall_score, f1_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback, pipeline
)
from datasets import Dataset
from lime.lime_text import LimeTextExplainer
import shap

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.facecolor'] = 'white'

DEVICE = 0 if torch.cuda.is_available() else -1

# ── Hindi-capable multilingual model ──────────────────────────────────────
# MuRIL is fine-tuned on 17 Indian languages including Hindi and is
# strongly preferred over mBERT for Devanagari text.
# Alternative: 'ai4bharat/indic-bert'  (also good for Hindi)
MODEL_NAME = 'google/muril-base-cased'

LOG_FILE   = 'results_log_uli_hindi.json'
SEED       = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Master results log (parallel structure to English log) ─────────────────
results_log = {
    'dataset':    'ULI-Hindi',
    'model_base': MODEL_NAME,
    'language':   'hi',
    'timestamp':  datetime.now().isoformat(),
    'experiments': {}
}

def save_log():
    with open(LOG_FILE, 'w') as f:
        json.dump(results_log, f, indent=2, ensure_ascii=False)
    print(f'Log saved → {LOG_FILE}')

print(f'Device : {"GPU" if DEVICE == 0 else "CPU"}')
print(f'Model  : {MODEL_NAME}')
print('Setup complete!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Device : GPU
Model  : google/muril-base-cased
Setup complete!


In [2]:
# ── 2. DOWNLOAD ULI DATASET ────────────────────────────────────────────────
!kaggle datasets download -d akshatwa99/uli-dataset -q
!unzip -q -o uli-dataset.zip
print('ULI dataset ready!')

Dataset URL: https://www.kaggle.com/datasets/akshatwa99/uli-dataset
License(s): unknown
ULI dataset ready!


In [ ]:
# ── 3. LOAD HINDI SPLITS ───────────────────────────────────────────────────
# The Hindi annotator columns follow the same pattern as English (hi_a1 … hi_a5)
# Majority-vote aggregation is identical to the English pipeline.

def load_hindi_split(paths):
    """
    Load multiple Hindi ULI CSV files and aggregate labels via majority voting.
    Annotator columns: hi_a1, hi_a2, hi_a3, hi_a4, hi_a5
    (5 annotators for Hindi vs 6 for English)
    """
    annotators = ['hi_a1', 'hi_a2', 'hi_a3', 'hi_a4', 'hi_a5']
    dfs  = [pd.read_csv(p, lineterminator='\n') for p in paths]
    base = dfs[0][['text']].copy()

    for col in annotators:
        for d in dfs:
            if col in d.columns:
                d[col] = pd.to_numeric(d[col], errors='coerce')
        # Only use files that actually contain this annotator column
        relevant = [d for d in dfs if col in d.columns]
        if not relevant:
            base[col] = np.nan
            continue
        stacked    = pd.concat([d[col] for d in relevant], axis=1)
        is_one     = stacked.eq(1).any(axis=1)
        is_all_nan = stacked.isna().all(axis=1)
        base[col]  = 0
        base.loc[is_one,     col] = 1
        base.loc[is_all_nan, col] = np.nan

    available_annotators = [c for c in annotators if c in base.columns]
    votes_mean             = base[available_annotators].mean(axis=1)
    base['hate_label']     = np.nan
    base.loc[votes_mean >= 0.5, 'hate_label'] = 1
    base.loc[votes_mean  < 0.5, 'hate_label'] = 0
    return base[['text', 'hate_label']].dropna(subset=['hate_label'])


BASE = '/content/uli_dataset-main'

df_train_raw = load_hindi_split([
    f'{BASE}/training/train_hi_l1.csv',
    f'{BASE}/training/train_hi_l2.csv',
    f'{BASE}/training/train_hi_l3.csv',
])
df_test_raw = load_hindi_split([
    f'{BASE}/testing/test_hi_l1.csv',
    f'{BASE}/testing/test_hi_l2.csv',
    f'{BASE}/testing/test_hi_l3.csv',
])

df_train_raw['hate_label'] = df_train_raw['hate_label'].astype(int)
df_test_raw['hate_label']  = df_test_raw['hate_label'].astype(int)

df_tr, df_val = train_test_split(
    df_train_raw, test_size=0.1, random_state=SEED,
    stratify=df_train_raw['hate_label']
)
df_te    = df_test_raw.copy()
df_total = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

print(f'Train : {len(df_tr):>5}  |  Val : {len(df_val):>4}  |  Test : {len(df_te):>4}')
print(f'Total : {len(df_total)}')
print('\nTest label distribution:')
print(df_te['hate_label'].value_counts().rename({0: 'Not Hate', 1: 'Hate'}))
print('\nSample Hindi texts:')
print(df_te['text'].head(3).tolist())

results_log['data_stats'] = {
    'train': len(df_tr), 'val': len(df_val), 'test': len(df_te),
    'test_hate':     int(df_te['hate_label'].sum()),
    'test_not_hate': int((df_te['hate_label'] == 0).sum())
}
save_log()

Train :  5577  |  Val :  620  |  Test : 1516
Total : 7713

Test label distribution:
hate_label
Hate        1009
Not Hate     507
Name: count, dtype: int64

Sample Hindi texts:
['#BandraStation #SharadPawar #Muradabad  अगर अभी आपको होश आ गया हो तो ये मान लीजिये की अब आप भारत के नागरिक नही है , अब आप आने वाले समय के शरिया कानून का पालन करते नजर आएंगे जहां काफिरों के लिए अलग क़ानून होगा और मुसलमानों के लिए अलग।  #गजवायेHind की शुरूआत हो चुकी है #10ईयर और।', '#ConspiracyAgainstIndia  सुन लो रे देश के गद्दारों घर के और बाहर वाले भी यदि तुम अभी भी नहीं सुधरे ना तो तुम्हारी गांड में छाता डालकर मोर बना दूंगा', '#MarathaReservation : महाराष्ट्र में जश्न का माहौल  #Devendrafadnavis #Marathas #Maratha #Reservation <handle replaced><handle replaced><handle replaced><handle replaced><handle replaced><handle replaced><handle replaced><handle replaced><handle replaced>  ']
Log saved → results_log_uli_hindi.json


In [ ]:
# ── 4. LOAD ZERO-SHOT MODEL (MuRIL) ───────────────────────────────────────
# MuRIL does NOT ship with a pre-built hate-speech head, so we run it
# zero-shot via a fill-mask / feature-extraction approach.
# For a true zero-shot classifier we use the HF zero-shot-classification
# pipeline with a multilingual NLI model instead.

ZS_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
print(f'Loading zero-shot NLI model: {ZS_MODEL} ...')

zeroshot_pipe = pipeline(
    'zero-shot-classification',
    model=ZS_MODEL,
    device=DEVICE,
    batch_size=32
)
CANDIDATE_LABELS = ['hate speech', 'not hate speech']
print('Zero-shot model loaded!')

Loading zero-shot NLI model: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7 ...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Zero-shot model loaded!


In [ ]:
# ── 5. HELPER FUNCTIONS ────────────────────────────────────────────────────

def run_zeroshot_pipeline(pipe, df, candidate_labels, batch_size=32):
    """Run zero-shot classification and return pred_label + confidence columns."""
    texts  = df['text'].tolist()
    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i:i + batch_size]
        results = pipe(batch, candidate_labels=candidate_labels, truncation=True)
        for r in results:
            top_label = r['labels'][0]
            top_score = r['scores'][0]
            preds.append(1 if top_label == 'hate speech' else 0)
            confs.append(round(top_score, 4))
    df = df.copy()
    df['pred_label'] = preds
    df['confidence'] = confs
    return df


def log_metrics(y_true, y_pred, exp_name, split='test'):
    """Compute, print and log all metrics."""
    acc         = accuracy_score(y_true, y_pred)
    f1          = f1_score(y_true, y_pred, average='macro')
    prec        = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec         = recall_score(y_true, y_pred, average='macro', zero_division=0)
    hate_recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    hate_prec   = precision_score(y_true, y_pred, pos_label=1, zero_division=0)

    print(f'\n===== {exp_name} | {split} =====')
    print(f'Accuracy      : {acc * 100:.2f}%')
    print(f'F1 (macro)    : {f1  * 100:.2f}%')
    print(f'Hate Recall   : {hate_recall * 100:.2f}%')
    print(f'Hate Precision: {hate_prec   * 100:.2f}%')
    print('\nFull Report:')
    print(classification_report(y_true, y_pred,
                                target_names=['Not Hate', 'Hate'],
                                zero_division=0))

    if exp_name not in results_log['experiments']:
        results_log['experiments'][exp_name] = {}
    results_log['experiments'][exp_name][split] = {
        'accuracy':        round(acc,         4),
        'f1_macro':        round(f1,          4),
        'precision_macro': round(prec,        4),
        'recall_macro':    round(rec,         4),
        'hate_recall':     round(hate_recall, 4),
        'hate_precision':  round(hate_prec,   4),
    }
    save_log()
    return results_log['experiments'][exp_name][split]


# Run zero-shot on test set
print('Running zero-shot inference on Hindi test set...')
df_te_zs = run_zeroshot_pipeline(zeroshot_pipe, df_te, CANDIDATE_LABELS)
zs_metrics = log_metrics(
    df_te_zs['hate_label'], df_te_zs['pred_label'],
    'muril_hindi_zeroshot'
)

Running zero-shot inference on Hindi test set...


KeyboardInterrupt: 

In [ ]:
# ── 6. LIME EXPLAINABILITY ─────────────────────────────────────────────────

def make_zs_predictor(pipe, candidate_labels):
    """Wrap zero-shot pipeline as a LIME-compatible predictor."""
    def predictor(texts):
        results = pipe(list(texts), candidate_labels=candidate_labels, truncation=True)
        probs   = []
        for r in results:
            hate_idx  = r['labels'].index('hate speech')
            hate_prob = r['scores'][hate_idx]
            probs.append([1 - hate_prob, hate_prob])
        return np.array(probs)
    return predictor


def make_ft_predictor(ft_pipe_obj, label_map):
    """Wrap fine-tuned text-classification pipeline as a LIME-compatible predictor."""
    def predictor(texts):
        results = ft_pipe_obj(list(texts), truncation=True)
        probs   = []
        for r in results:
            hate_score = r['score'] if label_map.get(r['label'], 0) == 1 else 1 - r['score']
            probs.append([1 - hate_score, hate_score])
        return np.array(probs)
    return predictor


def run_lime(predictor_fn, df, n_samples=50, label='zeroshot', save_prefix='hi_zs'):
    explainer   = LimeTextExplainer(class_names=['Not Hate', 'Hate'])
    hate_df     = df[(df['hate_label'] == 1) & (df['pred_label'] == 1)].reset_index(drop=True)
    texts       = hate_df['text'].iloc[:n_samples].tolist()
    word_scores = defaultdict(list)

    for i, text in enumerate(texts):
        if i % 10 == 0:
            print(f'  LIME {i}/{len(texts)}...')
        exp = explainer.explain_instance(text, predictor_fn,
                                         num_features=10, num_samples=300)
        for word, score in exp.as_list():
            word_scores[word].append(score)

    lime_df = pd.DataFrame(
        [(w, np.mean(s)) for w, s in word_scores.items()],
        columns=['word', 'avg_lime_score']
    ).sort_values('avg_lime_score', ascending=False).reset_index(drop=True)

    top15 = lime_df.head(15).sort_values('avg_lime_score')
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top15['word'], top15['avg_lime_score'], color='#E24B4A')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Avg LIME Score (toward Hate)', fontsize=11)
    ax.set_title(f'Top hate-driving words — {label}\n(ULI Hindi dataset)', fontsize=12)
    plt.tight_layout()
    fname = f'lime_{save_prefix}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

    key = label.replace(' ', '_')
    if key not in results_log['experiments']:
        results_log['experiments'][key] = {}
    results_log['experiments'][key]['lime_top15'] = (
        lime_df.head(15)[['word', 'avg_lime_score']]
        .assign(avg_lime_score=lambda x: x['avg_lime_score'].round(4))
        .to_dict(orient='records')
    )
    save_log()
    return lime_df


print('Running LIME on muril_hindi_zeroshot (50 samples, ~4-6 mins)...')
zs_predictor = make_zs_predictor(zeroshot_pipe, CANDIDATE_LABELS)
lime_zs = run_lime(zs_predictor, df_te_zs, label='muril_hindi_zeroshot', save_prefix='hi_zs')

In [ ]:
# ── 7. SHAP EXPLAINABILITY ─────────────────────────────────────────────────

def make_shap_predictor_zs(pipe, candidate_labels):
    def shap_pred(texts):
        results = pipe(list(texts), candidate_labels=candidate_labels, truncation=True)
        probs   = []
        for r in results:
            hate_idx = r['labels'].index('hate speech')
            probs.append(r['scores'][hate_idx])
        return np.array(probs)
    return shap_pred


def make_shap_predictor_ft(ft_pipe_obj, label_map):
    def shap_pred(texts):
        results = ft_pipe_obj(list(texts), truncation=True)
        return np.array([
            r['score'] if label_map.get(r['label'], 0) == 1 else 1 - r['score']
            for r in results
        ])
    return shap_pred


def run_shap(shap_pred_fn, df, n_samples=100, label='zeroshot', save_prefix='hi_zs'):
    print(f'Computing SHAP on {n_samples} samples (~6-8 mins)...')
    sample_texts = df.sample(n=min(n_samples, len(df)), random_state=SEED)['text'].tolist()
    masker       = shap.maskers.Text(r'\W+')
    explainer    = shap.Explainer(shap_pred_fn, masker)
    shap_values  = explainer(sample_texts, batch_size=16)

    shap.plots.bar(shap_values, max_display=20, show=False)
    plt.title(f'Top 20 words by SHAP value — {label}\n(ULI Hindi dataset)', fontsize=12)
    plt.tight_layout()
    fname = f'shap_{save_prefix}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')
    return shap_values


shap_zs_fn = make_shap_predictor_zs(zeroshot_pipe, CANDIDATE_LABELS)
shap_zs    = run_shap(shap_zs_fn, df_te_zs, label='muril_hindi_zeroshot', save_prefix='hi_zs')

In [ ]:
# ── 8. DEMOGRAPHIC BIAS ANALYSIS (HINDI KEYWORDS) ─────────────────────────
# Extended with Devanagari transliterations alongside romanised forms

DEMOGRAPHIC_KEYWORDS_HI = {
    'Gender': [
        # Romanised
        'aurat', 'ladki', 'mahila', 'beti', 'bitch', 'randi', 'besharmi',
        # Devanagari
        'औरत', 'लड़की', 'महिला', 'बेटी', 'रंडी',
    ],
    'Religion': [
        # Romanised
        'muslim', 'hindu', 'islam', 'allah', 'mandir', 'masjid', 'sikh',
        'kafir', 'jihad', 'christian', 'church',
        # Devanagari
        'मुस्लिम', 'हिंदू', 'इस्लाम', 'अल्लाह', 'मंदिर', 'मस्जिद',
        'काफिर', 'जिहाद', 'सिख', 'ईसाई',
    ],
    'Caste': [
        # Romanised
        'dalit', 'brahmin', 'untouchable', 'caste', 'sc', 'st', 'obc',
        'reservation', 'chamar', 'bhangi',
        # Devanagari
        'दलित', 'ब्राह्मण', 'अछूत', 'जाति', 'आरक्षण', 'चमार', 'भंगी',
    ],
    'Ethnicity / Region': [
        # Romanised
        'bangladeshi', 'rohingya', 'pakistani', 'nepali', 'bihari',
        'migrant', 'ghuspethiya', 'refugee',
        # Devanagari
        'बांग्लादेशी', 'रोहिंग्या', 'पाकिस्तानी', 'नेपाली', 'बिहारी',
        'घुसपैठिया', 'शरणार्थी',
    ],
}


def demographic_bias_analysis(df, exp_name, save_prefix,
                               keyword_dict=DEMOGRAPHIC_KEYWORDS_HI):
    rows = []
    for group, keywords in keyword_dict.items():
        mask   = df['text'].apply(
            lambda t: any(kw.lower() in str(t).lower() for kw in keywords)
        )
        subset = df[mask]
        if len(subset) == 0:
            continue
        true_hate = int(subset['hate_label'].sum())
        if true_hate == 0:
            continue
        hate_recall = round(
            subset[subset['hate_label'] == 1]['pred_label'].mean() * 100, 1
        )
        rows.append({
            'Group':         group,
            'Samples':       len(subset),
            'True Hate':     true_hate,
            'Pred Hate':     int(subset['pred_label'].sum()),
            'Hate Recall %': hate_recall,
        })

    bias_df = pd.DataFrame(rows)
    print(f'\nDemographic Bias — {exp_name}')
    print(bias_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    colors  = ['#E24B4A' if r < 60 else '#378ADD' for r in bias_df['Hate Recall %']]
    bars    = ax.bar(bias_df['Group'], bias_df['Hate Recall %'], color=colors)
    ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
    ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, label='50% baseline')
    ax.set_ylabel('Hate Recall (%)', fontsize=11)
    ax.set_ylim(0, 110)
    ax.set_title(f'Hate Recall by Demographic Group — {exp_name}\nRed = below 60%',
                 fontsize=12)
    ax.legend(fontsize=9)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    fname = f'demographic_{save_prefix}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

    results_log['experiments'][exp_name]['demographic_bias'] = \
        bias_df.to_dict(orient='records')
    save_log()
    return bias_df


bias_zs = demographic_bias_analysis(df_te_zs, 'muril_hindi_zeroshot', 'hi_zs')

In [ ]:
# ── 9. PERTURBATION TEST (HINDI SWAP PAIRS) ────────────────────────────────
# Swap pairs extended with romanised and Devanagari variants

SWAP_PAIRS_HI = [
    # Religion
    ('muslim',    'christian'), ('hindu',  'christian'),
    ('मुस्लिम',  'ईसाई'),      ('हिंदू',  'ईसाई'),
    # Gender
    ('aurat',     'aadmi'),     ('औरत',   'आदमी'),
    ('ladki',     'ladka'),     ('लड़की', 'लड़का'),
    # Caste
    ('dalit',     'person'),    ('दलित',   'व्यक्ति'),
]


def perturbation_test_zs(pipe, df, candidate_labels, exp_name, n=15):
    all_terms = [s for s, _ in SWAP_PAIRS_HI]
    pattern   = '|'.join(re.escape(t) for t in all_terms)
    candidates = df[df['text'].str.contains(pattern, na=False, case=False)].head(n)
    records    = []
    print(f'\nPerturbation Test — {exp_name}')
    print(f'{"Text[:55]":<57} | Orig | Swap | Flipped?')
    print('-' * 85)

    for _, row in candidates.iterrows():
        orig    = row['text']
        swapped = orig
        for src, tgt in SWAP_PAIRS_HI:
            swapped = re.sub(re.escape(src), tgt, swapped, flags=re.IGNORECASE)
        if swapped == orig:
            continue

        r_orig = pipe([orig],    candidate_labels=candidate_labels, truncation=True)[0]
        r_swap = pipe([swapped], candidate_labels=candidate_labels, truncation=True)[0]
        hi_idx = r_orig['labels'].index('hate speech')
        s_orig = r_orig['scores'][hi_idx]
        hi_idx = r_swap['labels'].index('hate speech')
        s_swap = r_swap['scores'][hi_idx]
        flipped = abs(s_orig - s_swap) > 0.2

        print(f'{orig[:55]:<57} | {s_orig:.2f} | {s_swap:.2f} | {"YES" if flipped else "no"}')
        records.append({
            'text': orig[:80],
            'orig_hate_score': round(s_orig, 4),
            'swap_hate_score': round(s_swap, 4),
            'flipped': flipped
        })

    flipped_count = sum(r['flipped'] for r in records)
    print(f'\nFlipped: {flipped_count}/{len(records)} changed by >0.2 after demographic swap')
    results_log['experiments'][exp_name]['perturbation'] = {
        'total_tested': len(records),
        'flipped':      flipped_count,
        'details':      records
    }
    save_log()


perturbation_test_zs(zeroshot_pipe, df_te_zs, CANDIDATE_LABELS, 'muril_hindi_zeroshot')

In [ ]:
# ── 10. SAVE ZERO-SHOT WEIGHTS ─────────────────────────────────────────────
zeroshot_pipe.model.save_pretrained('muril_hindi_zeroshot_model')
zeroshot_pipe.tokenizer.save_pretrained('muril_hindi_zeroshot_model')
print('Saved muril_hindi_zeroshot_model/')

In [ ]:
# ── 11. TOKENISE FOR FINE-TUNING ───────────────────────────────────────────
print(f'Loading MuRIL tokenizer from {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_df(df):
    ds = Dataset.from_pandas(
        df[['text', 'hate_label']]
        .rename(columns={'hate_label': 'label'})
        .reset_index(drop=True)
    )
    def tok(batch):
        return tokenizer(batch['text'],
                         padding='max_length',
                         truncation=True,
                         max_length=128)
    ds = ds.map(tok, batched=True)
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

train_ds = tokenize_df(df_tr)
val_ds   = tokenize_df(df_val)
test_ds  = tokenize_df(df_te)
print(f'Tokenised — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}')

In [ ]:
# ── 12. FINE-TUNE MuRIL ON HINDI ULI DATA ─────────────────────────────────
label_counts  = df_tr['hate_label'].value_counts().sort_index()
class_weights = torch.tensor(
    [len(df_tr) / (2 * c) for c in label_counts], dtype=torch.float
)
print(f'Class weights: Not Hate={class_weights[0]:.3f}  Hate={class_weights[1]:.3f}')

ft_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    round(accuracy_score(labels, preds), 4),
        'f1_macro':    round(f1_score(labels, preds, average='macro'), 4),
        'hate_recall': round(recall_score(labels, preds, pos_label=1, zero_division=0), 4),
    }

training_args = TrainingArguments(
    output_dir='./ft_uli_hindi_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=50,
    report_to='none',
    push_to_hub=False,
)

trainer = WeightedTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print('Trainer ready!')

In [ ]:
# ── 13. TRAIN ──────────────────────────────────────────────────────────────
print('Starting fine-tuning (~15-20 mins on T4 x2)...')
train_result = trainer.train()
print('\nFine-tuning complete!')
print(train_result.metrics)

In [ ]:
# ── 14. TRAINING CURVES ────────────────────────────────────────────────────
log_history  = trainer.state.log_history
train_losses = [(e['epoch'], e['loss'])
                for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_metrics = [(e['epoch'], e['eval_loss'],
                 e.get('eval_f1_macro', 0), e.get('eval_hate_recall', 0))
                for e in log_history if 'eval_loss' in e]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
te, tl = zip(*train_losses)
axes[0].plot(te, tl, color='#378ADD')
axes[0].set_title('Training Loss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)

ee, el, ef, er = zip(*eval_metrics)
axes[1].plot(ee, el, color='#E24B4A', marker='o')
axes[1].set_title('Validation Loss', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].grid(alpha=0.3)

axes[2].plot(ee, ef, color='#1D9E75', marker='o', label='F1 macro')
axes[2].plot(ee, er, color='#E24B4A', marker='s', label='Hate Recall')
axes[2].set_title('Val F1 & Hate Recall', fontsize=11)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.suptitle('Training Curves — muril_hindi_finetuned', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves_hindi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves_hindi.png')

results_log['experiments']['muril_hindi_finetuned'] = {
    'train_metrics': train_result.metrics,
    'training_curves': {
        'train_loss': list(zip([round(e, 2) for e in te], [round(l, 4) for l in tl])),
        'eval_loss':  list(zip([round(e, 2) for e in ee], [round(l, 4) for l in el])),
        'eval_f1':    list(zip([round(e, 2) for e in ee], [round(f, 4) for f in ef])),
    }
}
save_log()

In [ ]:
# ── 15. EVALUATE FINE-TUNED MODEL ON TEST SET ──────────────────────────────
ft_preds_raw = trainer.predict(test_ds)
ft_preds     = np.argmax(ft_preds_raw.predictions, axis=-1)
y_true_test  = df_te['hate_label'].tolist()
df_te_ft     = df_te.copy()
df_te_ft['pred_label'] = ft_preds

ft_metrics = log_metrics(y_true_test, ft_preds, 'muril_hindi_finetuned')

In [ ]:
# ── 16. LIME ON FINE-TUNED MODEL ───────────────────────────────────────────
# Build a text-classification pipeline from the fine-tuned model
# id2label: 0 = LABEL_0 (Not Hate), 1 = LABEL_1 (Hate)
ft_pipe = pipeline(
    'text-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    device=DEVICE,
    batch_size=64
)
# Map model output labels to 0/1
ft_label_map = {'LABEL_0': 0, 'LABEL_1': 1}

print('Running LIME on muril_hindi_finetuned (50 samples, ~4 mins)...')
ft_predictor = make_ft_predictor(ft_pipe, ft_label_map)
lime_ft = run_lime(ft_predictor, df_te_ft, label='muril_hindi_finetuned', save_prefix='hi_ft')

In [ ]:
# ── 17. SHAP ON FINE-TUNED MODEL ───────────────────────────────────────────
shap_ft_fn = make_shap_predictor_ft(ft_pipe, ft_label_map)
shap_ft    = run_shap(shap_ft_fn, df_te_ft, label='muril_hindi_finetuned', save_prefix='hi_ft')

In [ ]:
# ── 18. DEMOGRAPHIC BIAS — FINE-TUNED ─────────────────────────────────────
bias_ft = demographic_bias_analysis(df_te_ft, 'muril_hindi_finetuned', 'hi_ft')

In [ ]:
# ── 19. PERTURBATION TEST — FINE-TUNED ────────────────────────────────────

def perturbation_test_ft(pipe, df, label_map, exp_name, n=15):
    all_terms  = [s for s, _ in SWAP_PAIRS_HI]
    pattern    = '|'.join(re.escape(t) for t in all_terms)
    candidates = df[df['text'].str.contains(pattern, na=False, case=False)].head(n)
    records    = []
    print(f'\nPerturbation Test — {exp_name}')
    print(f'{"Text[:55]":<57} | Orig | Swap | Flipped?')
    print('-' * 85)

    for _, row in candidates.iterrows():
        orig    = row['text']
        swapped = orig
        for src, tgt in SWAP_PAIRS_HI:
            swapped = re.sub(re.escape(src), tgt, swapped, flags=re.IGNORECASE)
        if swapped == orig:
            continue

        r_orig = pipe([orig],    truncation=True)[0]
        r_swap = pipe([swapped], truncation=True)[0]
        s_orig = r_orig['score'] if label_map.get(r_orig['label'], 0) == 1 else 1 - r_orig['score']
        s_swap = r_swap['score'] if label_map.get(r_swap['label'], 0) == 1 else 1 - r_swap['score']
        flipped = abs(s_orig - s_swap) > 0.2

        print(f'{orig[:55]:<57} | {s_orig:.2f} | {s_swap:.2f} | {"YES" if flipped else "no"}')
        records.append({
            'text': orig[:80],
            'orig_hate_score': round(s_orig, 4),
            'swap_hate_score': round(s_swap, 4),
            'flipped': flipped
        })

    flipped_count = sum(r['flipped'] for r in records)
    print(f'\nFlipped: {flipped_count}/{len(records)} changed by >0.2 after demographic swap')
    results_log['experiments'][exp_name]['perturbation'] = {
        'total_tested': len(records),
        'flipped':      flipped_count,
        'details':      records
    }
    save_log()


perturbation_test_ft(ft_pipe, df_te_ft, ft_label_map, 'muril_hindi_finetuned')

In [ ]:
# ── 20. SAVE FINE-TUNED WEIGHTS ────────────────────────────────────────────
trainer.model.save_pretrained('muril_hindi_finetuned_model')
tokenizer.save_pretrained('muril_hindi_finetuned_model')
print('Saved muril_hindi_finetuned_model/')

In [ ]:
# ── 21. ZERO-SHOT vs FINE-TUNED COMPARISON ────────────────────────────────
exp = results_log['experiments']
models       = ['muril_hindi_zeroshot', 'muril_hindi_finetuned']
labels_plot  = ['Zero-shot (mDeBERTa NLI)', 'Fine-tuned (MuRIL)']
metric_keys  = ['accuracy', 'f1_macro', 'hate_recall', 'hate_precision']
metric_labels = ['Accuracy', 'F1 (macro)', 'Hate Recall', 'Hate Precision']
colors       = ['#888780', '#1D9E75']

x = np.arange(len(metric_keys)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
for i, (m, lbl, c) in enumerate(zip(models, labels_plot, colors)):
    vals = [exp[m]['test'][k] * 100 for k in metric_keys]
    bars = ax.bar(x + (i - 0.5) * w, vals, w, label=lbl, color=c)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel('Score (%)', fontsize=11); ax.set_ylim(0, 105)
ax.set_title('ULI Hindi: Zero-shot vs Fine-tuned', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_hindi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved comparison_hindi.png')

In [ ]:
# ── 22. LIME COMPARISON PLOT ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

top_zs = lime_zs.head(15).sort_values('avg_lime_score')
ax1.barh(top_zs['word'], top_zs['avg_lime_score'], color='#888780')
ax1.set_title('Top hate words — Zero-shot', fontsize=12)
ax1.set_xlabel('Avg LIME Score')
ax1.axvline(0, color='gray', linewidth=0.8, linestyle='--')

top_ft = lime_ft.head(15).sort_values('avg_lime_score')
ax2.barh(top_ft['word'], top_ft['avg_lime_score'], color='#1D9E75')
ax2.set_title('Top hate words — Fine-tuned', fontsize=12)
ax2.set_xlabel('Avg LIME Score')
ax2.axvline(0, color='gray', linewidth=0.8, linestyle='--')

plt.suptitle('Did fine-tuning change what words drive hate predictions? (LIME)\nULI Hindi Dataset',
             fontsize=13)
plt.tight_layout()
plt.savefig('lime_comparison_hindi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved lime_comparison_hindi.png')

In [ ]:
# ── 23. DEMOGRAPHIC BIAS COMPARISON PLOT ──────────────────────────────────
bias_zs_r = bias_zs.set_index('Group')['Hate Recall %']
bias_ft_r = bias_ft.set_index('Group')['Hate Recall %']
groups    = bias_zs_r.index.tolist()

x = np.arange(len(groups)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w / 2, bias_zs_r.values, w, label='Zero-shot', color='#888780')
b2 = ax.bar(x + w / 2, bias_ft_r.reindex(groups).values, w, label='Fine-tuned', color='#1D9E75')
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Hate Recall (%)', fontsize=11); ax.set_ylim(0, 115)
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Demographic Bias: Did fine-tuning reduce it?\nULI Hindi Dataset', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('demographic_comparison_hindi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved demographic_comparison_hindi.png')

In [ ]:
# ── 24. FINAL SUMMARY ─────────────────────────────────────────────────────
save_log()
print('\n===== FINAL RESULTS SUMMARY — ULI HINDI =====')
for exp_name, exp_data in results_log['experiments'].items():
    if 'test' in exp_data:
        t = exp_data['test']
        print(f'\n{exp_name}')
        print(f'  Accuracy      : {t["accuracy"]      * 100:.2f}%')
        print(f'  F1 (macro)    : {t["f1_macro"]       * 100:.2f}%')
        print(f'  Hate Recall   : {t["hate_recall"]    * 100:.2f}%')
        print(f'  Hate Precision: {t["hate_precision"] * 100:.2f}%')

print('\nAll outputs saved:')
print('  results_log_uli_hindi.json')
print('  muril_hindi_zeroshot_model/   muril_hindi_finetuned_model/')
print('  lime_hi_zs.png    | lime_hi_ft.png    | lime_comparison_hindi.png')
print('  shap_hi_zs.png    | shap_hi_ft.png')
print('  demographic_hi_zs.png | demographic_hi_ft.png | demographic_comparison_hindi.png')
print('  training_curves_hindi.png | comparison_hindi.png')
print('\nTo compare English vs Hindi results, load both JSON logs:')
print('  en_log = json.load(open("results_log_uli.json"))')
print('  hi_log = json.load(open("results_log_uli_hindi.json"))')